In [2]:
import os
import pandas as pd
import regex as re
import logging
import numpy as np
from sklearn.linear_model import BayesianRidge
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import json
from datetime import timedelta

In [3]:
def infer_campaign_type(campaign_name):
    if not isinstance(campaign_name, str):
        return "Other"
    cn = campaign_name.lower()
    if re.search(r'brand|branded|netelixir', cn):
        return "Brand Search"
    elif re.search(r'search|keyword', cn):
        return "Non-Brand Search"
    elif re.search(r'shopping|pla|merchant', cn):
        return "Shopping"
    elif re.search(r'pmax|performance\.max', cn):
        return "PMAX"
    elif re.search(r'retarg|remarketing|rlsa', cn):
        return "Retargeting"
    elif re.search(r'display|gdn|audience', cn):
        return "Display"
    elif re.search(r'video|youtube|reels|vma', cn):
        return "Video"
    else:
        return "Other"

In [4]:
def deduplicate_and_select(df):
    if df.empty: return df
    df = df.sort_values('date')
    df = df.drop_duplicates(subset=['date', 'channel', 'campaign_name'], keep='last')
    mask = (df['spend'] == 0) & (df['conversion_value'] == 0) & (df['impressions'] == 0)
    df = df[~mask]
    cols = ['date', 'channel', 'campaign_type', 'campaign_name', 'spend', 'impressions', 'clicks', 'conversions', 'conversion_value']
    return df[cols]

In [5]:
def load_google_ads(filepath):
    if not os.path.exists(filepath): return pd.DataFrame()
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['segments_date'])
    df['channel'] = 'google'
    df['campaign_type'] = df['campaign_name'].apply(infer_campaign_type)
    df['spend'] = df['metrics_cost_micros'] / 1e6
    df['impressions'] = df['metrics_impressions']
    df['clicks'] = df['metrics_clicks']
    df['conversions'] = df['metrics_conversions']
    df['conversion_value'] = df['metrics_conversions_value']
    return deduplicate_and_select(df)

In [6]:
def load_meta_ads(filepath):
    if not os.path.exists(filepath): return pd.DataFrame()
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date_start'])
    df['channel'] = 'meta'
    df['campaign_type'] = df['campaign_name'].apply(infer_campaign_type)
    df['spend'] = df['spend']
    df['impressions'] = df['impressions']
    df['clicks'] = df['clicks']
    df['conversions'] = 0.0
    df['conversion_value'] = df['conversion']
    return deduplicate_and_select(df)

In [7]:
def load_microsoft_ads(filepath):
    if not os.path.exists(filepath): return pd.DataFrame()
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['TimePeriod'])
    df['channel'] = 'microsoft'
    df['campaign_type'] = df['CampaignName'].apply(infer_campaign_type)
    df['campaign_name'] = df['CampaignName']
    df['spend'] = df['Spend']
    df['impressions'] = df['Impressions']
    df['clicks'] = df['Clicks']
    df['conversions'] = df['Conversions']
    df['conversion_value'] = df['Revenue']
    return deduplicate_and_select(df)

In [8]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [9]:
logging.info("Ingesting raw CSVs...")
df_google = load_google_ads('google_ads_campaign_stats.csv')
df_meta = load_meta_ads('meta_ads_campaign_stats.csv')
df_ms = load_microsoft_ads('bing_campaign_stats.csv')

INFO: Ingesting raw CSVs...


In [10]:
df_all = pd.concat([df_google, df_meta, df_ms], ignore_index=True)

In [11]:
df_google.head()

,date,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value
3183,2024-01-01,google,PMAX,Pmax_Campaign_07,56.345718,5564,133,1.000000,160.510000
3404,2024-01-01,google,PMAX,Pmax_Campaign_08,53.343372,19543,145,1.700000,171.235000
2799,2024-01-01,google,PMAX,Pmax_Campaign_06,62.358381,20532,252,4.021899,303.804024
997,2024-01-01,google,PMAX,Pmax_NTM_Campaign_10,428.540565,163705,1520,24.266729,3382.542613
883,2024-01-01,google,PMAX,Pmax_Campaign_03,7.162371,2687,28,0.987045,19.731030


In [12]:
df_meta.head()

,date,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value
0,2024-05-23,meta,Other,Generic_Campaign_02,85.0,5188.0,37.0,0.0,0.0
1,2024-05-24,meta,Other,Generic_Campaign_02,85.0,5080.0,38.0,0.0,183.0
2,2024-05-25,meta,Other,Generic_Campaign_02,85.0,4627.0,19.0,0.0,163.2
3,2024-05-26,meta,Other,Generic_Campaign_02,85.0,4282.0,46.0,0.0,0.0
4,2024-05-27,meta,Other,Generic_Campaign_02,85.0,6355.0,57.0,0.0,206.4


In [13]:
df_ms.head()

,date,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value
0,2024-05-25,microsoft,Non-Brand Search,Search_TM_Campaign_02,4.70,140,22,0.0,0.0
1,2024-05-26,microsoft,Non-Brand Search,Search_TM_Campaign_02,4.30,120,14,0.0,0.0
2,2024-05-27,microsoft,Non-Brand Search,Search_TM_Campaign_02,4.41,149,18,0.0,0.0
3,2024-05-28,microsoft,Non-Brand Search,Search_TM_Campaign_02,2.61,135,8,0.0,0.0
4,2024-05-29,microsoft,Non-Brand Search,Search_TM_Campaign_02,2.56,145,10,0.0,0.0


In [14]:
df_all.head()

,date,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value
0,2024-01-01,google,PMAX,Pmax_Campaign_07,56.345718,5564.0,133.0,1.000000,160.510000
1,2024-01-01,google,PMAX,Pmax_Campaign_08,53.343372,19543.0,145.0,1.700000,171.235000
2,2024-01-01,google,PMAX,Pmax_Campaign_06,62.358381,20532.0,252.0,4.021899,303.804024
3,2024-01-01,google,PMAX,Pmax_NTM_Campaign_10,428.540565,163705.0,1520.0,24.266729,3382.542613
4,2024-01-01,google,PMAX,Pmax_Campaign_03,7.162371,2687.0,28.0,0.987045,19.731030


In [15]:
def pad_missing_weeks(weekly):
    if weekly.empty: return weekly
    min_date = weekly['week_start'].min()
    max_date = weekly['week_start'].max()
    all_weeks = pd.date_range(min_date, max_date, freq='W-MON')

    padded_list = []
    for (ch, ct, cn), grp in weekly.groupby(['channel', 'campaign_type', 'campaign_name']):
        grp = grp.set_index('week_start')
        grp = grp.reindex(all_weeks)
        grp['channel'] = ch
        grp['campaign_type'] = ct
        grp['campaign_name'] = cn
        grp.fillna({
            'spend': 0, 'impressions': 0, 'clicks': 0, 'conversions': 0, 'conversion_value': 0
        }, inplace=True)
        padded_list.append(grp.reset_index(names='week_start'))

    return pd.concat(padded_list, ignore_index=True)

def aggregate_to_weekly(df):
    if df.empty: return df
    df['week_start'] = df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='d')
    agg_funcs = {
        'spend': 'sum',
        'impressions': 'sum',
        'clicks': 'sum',
        'conversions': 'sum',
        'conversion_value': 'sum'
    }
    weekly = df.groupby(['week_start', 'channel', 'campaign_type', 'campaign_name'], as_index=False).agg(agg_funcs)
    weekly = pad_missing_weeks(weekly)
    weekly['roas_reported'] = np.where(weekly['spend'] > 0, weekly['conversion_value'] / weekly['spend'], 0)
    return weekly

In [16]:
weekly_df = aggregate_to_weekly(df_all)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10760\2140166517.py:23: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df['week_start'] = df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='d')


In [17]:
weekly_df.head(50)

,week_start,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value,roas_reported
0,2024-01-01,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-01-08,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-01-15,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-01-22,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-01-29,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
5,2024-02-05,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
6,2024-02-12,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
7,2024-02-19,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
8,2024-02-26,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0
9,2024-03-04,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
import numpy as np
import pandas as pd

def compute_features(weekly_df):
    if weekly_df.empty: return weekly_df

    weekly_df = weekly_df.sort_values(['channel', 'campaign_type', 'campaign_name', 'week_start'])

    weekly_df['CPC'] = np.where(weekly_df['clicks'] > 0, weekly_df['spend'] / weekly_df['clicks'], 0)
    weekly_df['CVR'] = np.where(weekly_df['clicks'] > 0, weekly_df['conversions'] / weekly_df['clicks'], 0)
    weekly_df['AOV'] = np.where(weekly_df['conversions'] > 0, weekly_df['conversion_value'] / weekly_df['conversions'], 0)
    weekly_df['ROAS'] = weekly_df['roas_reported']

    weekly_total_spend = weekly_df.groupby('week_start')['spend'].transform('sum')
    weekly_channel_spend = weekly_df.groupby(['week_start', 'channel'])['spend'].transform('sum')
    weekly_df['spend_share'] = np.where(weekly_total_spend > 0, weekly_channel_spend / weekly_total_spend, 0)

    weekly_df['log_spend'] = np.log1p(weekly_df['spend'])
    weekly_df['log_revenue'] = np.log1p(weekly_df['conversion_value'])

    weekly_df['week_of_year'] = weekly_df['week_start'].dt.isocalendar().week
    weekly_df['sin_week'] = np.sin(2 * np.pi * weekly_df['week_of_year'] / 52)
    weekly_df['cos_week'] = np.cos(2 * np.pi * weekly_df['week_of_year'] / 52)

    holiday_weeks = {47, 48, 51, 52, 1, 7, 21, 36, 28}
    weekly_df['holiday_flag'] = weekly_df['week_of_year'].isin(holiday_weeks).astype(int)
    weekly_df['pre_holiday_week'] = weekly_df['week_of_year'].apply(lambda w: 1 if (w + 1) in holiday_weeks else 0)
    weekly_df['post_holiday_week'] = weekly_df['week_of_year'].apply(lambda w: 1 if (w - 1) in holiday_weeks else 0)

    grouped = weekly_df.groupby(['channel', 'campaign_type'])

    weekly_df['lag_1w_revenue'] = grouped['conversion_value'].shift(1).fillna(0)
    weekly_df['lag_2w_revenue'] = grouped['conversion_value'].shift(2).fillna(0)
    weekly_df['lag_4w_revenue'] = grouped['conversion_value'].shift(4).fillna(0)

    weekly_df['rolling_4w_avg_revenue'] = grouped['conversion_value'].transform(lambda x: x.rolling(4, min_periods=1).mean()).fillna(0)
    weekly_df['rolling_4w_std_revenue'] = grouped['conversion_value'].transform(lambda x: x.rolling(4, min_periods=1).std()).fillna(0)
    weekly_df['rolling_4w_avg_ROAS'] = grouped['ROAS'].transform(lambda x: x.rolling(4, min_periods=1).mean()).fillna(0)
    weekly_df['rolling_4w_avg_CVR'] = grouped['CVR'].transform(lambda x: x.rolling(4, min_periods=1).mean()).fillna(0)

    weekly_df['efficiency_index'] = np.where(weekly_df['rolling_4w_avg_ROAS'] > 0,
                                             weekly_df['ROAS'] / weekly_df['rolling_4w_avg_ROAS'], 0)

    return weekly_df


In [19]:
features_df = compute_features(weekly_df)

In [20]:
features_df.head(50)

,week_start,channel,campaign_type,campaign_name,spend,impressions,clicks,conversions,conversion_value,roas_reported,...,pre_holiday_week,post_holiday_week,lag_1w_revenue,lag_2w_revenue,lag_4w_revenue,rolling_4w_avg_revenue,rolling_4w_std_revenue,rolling_4w_avg_ROAS,rolling_4w_avg_CVR,efficiency_index
0,2024-01-01,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-01-08,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-01-15,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-01-22,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-01-29,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2024-02-05,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,2024-02-12,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2024-02-19,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2024-02-26,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2024-03-04,google,Non-Brand Search,Search_Campaign_01,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:

class BayesianForecaster:
    def __init__(self):
        self.model = BayesianRidge(compute_score=True)
        self.features = ['log_spend', 'sin_week', 'cos_week', 'lag_1w_revenue', 'lag_4w_revenue',
                         'rolling_4w_avg_ROAS', 'rolling_4w_avg_CVR', 'holiday_flag',
                         'pre_holiday_week', 'post_holiday_week']

    def fit(self, df):
        if len(df) < 8:
            return False
        X = df[self.features].fillna(0).astype(float).values
        y = df['log_revenue'].astype(float).values
        self.model.fit(X, y)
        return True

    def predict(self, X_future, n_samples=5000):
        X = X_future[self.features].fillna(0).astype(float).values
        mu_pred, sigma_pred = self.model.predict(X, return_std=True)

        samples = np.random.normal(mu_pred, sigma_pred, size=(n_samples, len(mu_pred))).T
        rev_samples = np.expm1(samples)

        p10 = np.percentile(rev_samples, 10, axis=1)
        p50 = np.percentile(rev_samples, 50, axis=1)
        p90 = np.percentile(rev_samples, 90, axis=1)

        return p10, p50, p90, rev_samples

class SeasonalNaiveForecaster:
    def __init__(self):
        self.overall_avg = 0
        self.month_avgs = {}

    def fit(self, df):
        self.overall_avg = df['conversion_value'].mean()
        df_copy = df.copy()
        df_copy['month'] = df_copy['week_start'].dt.month
        self.month_avgs = df_copy.groupby('month')['conversion_value'].mean().to_dict()

    def predict(self, X_future, df_hist):
        last_4w_avg = df_hist['conversion_value'].tail(4).mean()
        last_4w_cv = df_hist['conversion_value'].tail(4).std() / (last_4w_avg + 1e-5)
        last_4w_cv = 0 if pd.isna(last_4w_cv) else last_4w_cv

        p10s, p50s, p90s = [], [], []
        rev_samples_list = []

        for _, row in X_future.iterrows():
            m = row['week_start'].month
            seasonal_index = self.month_avgs.get(m, self.overall_avg) / (self.overall_avg + 1e-5) if self.overall_avg > 0 else 1.0
            p50 = last_4w_avg * seasonal_index

            p10 = p50 * (1 - max(0.15, 1.5 * last_4w_cv))
            p90 = p50 * (1 + max(0.15, 1.5 * last_4w_cv))

            p10 = max(0, p10)

            p10s.append(p10)
            p50s.append(p50)
            p90s.append(p90)

            samples = np.random.normal(p50, (p90-p10)/3.29 + 1e-5, size=5000)
            samples = np.maximum(samples, 0)
            rev_samples_list.append(samples)

        return np.array(p10s), np.array(p50s), np.array(p90s), np.array(rev_samples_list)


In [22]:
class ResponseCurve:
    def __init__(self):
        self.a = 0
        self.b = 0.75 # Default elasticity

    def fit(self, df):
        df_valid = df[(df['spend'] > 0) & (df['conversion_value'] > 0)]
        if len(df_valid) < 8:
            if len(df_valid) > 0:
                self.a = np.mean(df_valid['conversion_value']) / (np.mean(df_valid['spend'])**self.b + 1e-5)
            return False

        y = np.log(df_valid['conversion_value'])
        X = np.log(df_valid['spend'])
        X = add_constant(X)

        try:
            model = OLS(y, X).fit()
            if 'spend' in model.params:
                self.b = model.params['spend']
                self.a = np.exp(model.params['const'])

            if self.b >= 1.0 or self.b <= 0:
                self.b = 0.75
                self.a = np.mean(df_valid['conversion_value']) / (np.mean(df_valid['spend'])**self.b + 1e-5)
            return True
        except Exception:
            if len(df_valid) > 0:
                self.a = np.mean(df_valid['conversion_value']) / (np.mean(df_valid['spend'])**self.b + 1e-5)
            return False

    def predict_revenue(self, spend):
        return self.a * (spend ** self.b)

def generate_scenarios(current_spend_pace):
    return {
        'Conservative': 0.8 * current_spend_pace,
        'Baseline': 1.0 * current_spend_pace,
        'Aggressive': 1.2 * current_spend_pace
    }

In [23]:
segments = features_df.groupby(['channel', 'campaign_type'])
models = {}
curves = {}

for (ch, ct), grp in segments:
    bf = BayesianForecaster()
    if not bf.fit(grp):
        snf = SeasonalNaiveForecaster()
        snf.fit(grp)
        models[(ch, ct)] = ('snf', snf, grp)
    else:
        models[(ch, ct)] = ('bayesian', bf, grp)

    rc = ResponseCurve()
    rc.fit(grp)
    curves[(ch, ct)] = rc
logging.info("Forecast generation (Baseline scenario, 30 days)...")

forecast_results = {
    "forecast_period_days": 30,
    "scenario": "Baseline",
    "channels": {}
}

total_revenue_samples = np.zeros((5000,))
total_budget_input = 0.0

for (ch, ct), (mod_type, model, grp) in models.items():
    if ch not in forecast_results['channels']:
        forecast_results['channels'][ch] = {
            "budget_allocated": 0.0,
            "revenue": {"P10": 0.0, "P50": 0.0, "P90": 0.0},
            "roas": {"P10": 0.0, "P50": 0.0, "P90": 0.0},
            "campaign_types": []
        }

    # simple baseline: project last 4 weeks avg spend
    recent_avg_weekly_spend = grp['spend'].tail(4).mean()
    forecast_spend = recent_avg_weekly_spend * 4.3  # approx 30 days
    if pd.isna(forecast_spend): forecast_spend = 0.0

    rc = curves[(ch, ct)]
    expected_rev = rc.predict_revenue(forecast_spend)

    # We simulate the 30-day ahead by passing 4 empty future weeks to model.predict
    # In this hackathon prototype we'll do a simple projection using the models
    future_dates = pd.date_range(grp['week_start'].max() + timedelta(days=7), periods=4, freq='W-MON')
    future_df = pd.DataFrame({'week_start': future_dates})

    if mod_type == 'bayesian':
        future_df['log_spend'] = np.log1p(forecast_spend / 4.3)
        future_df['sin_week'] = np.sin(2 * np.pi * future_df['week_start'].dt.isocalendar().week / 52)
        future_df['cos_week'] = np.cos(2 * np.pi * future_df['week_start'].dt.isocalendar().week / 52)
        for f in ['lag_1w_revenue', 'lag_4w_revenue', 'rolling_4w_avg_ROAS', 'rolling_4w_avg_CVR', 'holiday_flag', 'pre_holiday_week', 'post_holiday_week']:
            future_df[f] = grp[f].iloc[-1] if not grp.empty else 0
        p10, p50, p90, samples = model.predict(future_df, n_samples=5000)
        # sum over the 4 weeks
        segment_rev_samples = np.sum(samples, axis=0)
    else:
        p10, p50, p90, samples = model.predict(future_df, grp)
        segment_rev_samples = np.sum(samples, axis=0)

    p10_val = max(0, np.percentile(segment_rev_samples, 10))
    p50_val = max(0, np.percentile(segment_rev_samples, 50))
    p90_val = max(0, np.percentile(segment_rev_samples, 90))

    # update channel aggregation
    forecast_results['channels'][ch]['budget_allocated'] += forecast_spend
    forecast_results['channels'][ch]['revenue']['P10'] += p10_val
    forecast_results['channels'][ch]['revenue']['P50'] += p50_val
    forecast_results['channels'][ch]['revenue']['P90'] += p90_val

    total_budget_input += forecast_spend
    total_revenue_samples += segment_rev_samples

    forecast_results['channels'][ch]['campaign_types'].append({
        "campaign_type": ct,
        "budget_allocated": round(forecast_spend, 2),
        "revenue": {
            "P10": round(p10_val, 2),
            "P50": round(p50_val, 2),
            "P90": round(p90_val, 2)
        },
        "roas": {
            "P10": round(p10_val / forecast_spend, 2) if forecast_spend > 0 else 0,
            "P50": round(p50_val / forecast_spend, 2) if forecast_spend > 0 else 0,
            "P90": round(p90_val / forecast_spend, 2) if forecast_spend > 0 else 0
        }
    })
forecast_results['total_budget_input'] = round(total_budget_input, 2)
forecast_results['revenue'] = {
    "P10": round(np.percentile(total_revenue_samples, 10), 2),
    "P50": round(np.percentile(total_revenue_samples, 50), 2),
    "P90": round(np.percentile(total_revenue_samples, 90), 2),
    "currency": "USD"
}
forecast_results['blended_roas'] = {
    "P10": round(forecast_results['revenue']['P10'] / total_budget_input, 2) if total_budget_input > 0 else 0,
    "P50": round(forecast_results['revenue']['P50'] / total_budget_input, 2) if total_budget_input > 0 else 0,
    "P90": round(forecast_results['revenue']['P90'] / total_budget_input, 2) if total_budget_input > 0 else 0
}
# Channel level roas
for ch in forecast_results['channels']:
    c_bud = forecast_results['channels'][ch]['budget_allocated']
    if c_bud > 0:
        forecast_results['channels'][ch]['roas']['P10'] = round(forecast_results['channels'][ch]['revenue']['P10'] / c_bud, 2)
        forecast_results['channels'][ch]['roas']['P50'] = round(forecast_results['channels'][ch]['revenue']['P50'] / c_bud, 2)
        forecast_results['channels'][ch]['roas']['P90'] = round(forecast_results['channels'][ch]['revenue']['P90'] / c_bud, 2)

    forecast_results['channels'][ch]['budget_allocated'] = round(c_bud, 2)
    for k in ['P10', 'P50', 'P90']:
        forecast_results['channels'][ch]['revenue'][k] = round(forecast_results['channels'][ch]['revenue'][k], 2)
with open('forecast_output.json', 'w') as f:
    json.dump(forecast_results, f, indent=2)
logging.info("Forecasting pipeline complete. Output saved to forecast_output.json.")

INFO: Forecast generation (Baseline scenario, 30 days)...
INFO: Forecasting pipeline complete. Output saved to forecast_output.json.
